In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df_data = pd.read_csv('datasets/opentender_clean_2022_2024.csv')

In [5]:
df_data.head()

,ocid,buyer_name,vendor_name,HPS,contract_value,procurement_method,category,item_description,date
0,ocds-20h3g7-12482010,Pemerintah Daerah Kota Surabaya,PT. DUTA BHUANA JAYA,7.445028e+08,6.666604e+08,consultancyServices,services,Jasa Konsultansi Badan Usaha Konstruksi,2024-02-02
1,ocds-20h3g7-12487010,Pemerintah Daerah Kota Surabaya,"MARGA PERKASA,CV",4.793422e+09,4.129909e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
2,ocds-20h3g7-12492010,Pemerintah Daerah Kota Surabaya,CV. Naga Kencana Wiratama,2.255991e+09,2.012700e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15
3,ocds-20h3g7-12496010,Pemerintah Daerah Kota Surabaya,CV. Tiga Points Jaya Karya,3.425503e+09,2.566207e+09,NaN,works,Pekerjaan Konstruksi,2024-01-16
4,ocds-20h3g7-12504010,Pemerintah Daerah Kota Surabaya,CV. Citra Karya,1.307263e+09,1.231890e+09,NaN,works,Pekerjaan Konstruksi,2024-01-15


In [4]:
df_data.head()
df_data.info()
df_data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 292852 entries, 0 to 292851
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ocid                292852 non-null  str    
 1   buyer_name          288191 non-null  str    
 2   vendor_name         164113 non-null  str    
 3   HPS                 163738 non-null  float64
 4   contract_value      164113 non-null  float64
 5   procurement_method  26328 non-null   str    
 6   category            164113 non-null  str    
 7   item_description    164113 non-null  str    
 8   date                292852 non-null  str    
dtypes: float64(2), str(7)
memory usage: 48.0 MB


,HPS,contract_value
count,1.637380e+05,1.641130e+05
mean,3.817390e+09,3.529409e+09
std,3.461689e+10,3.307242e+10
min,1.760000e+04,1.050000e+04
25%,3.795271e+08,3.504016e+08
50%,6.947567e+08,6.399824e+08
75%,1.800000e+09,1.660123e+09
max,4.286669e+12,4.242926e+12


*CLEANING NOISE*

In [7]:
df_data = df_data.drop(columns=["procurement_method"])
#kudrop noise tinggi

In [8]:
df_data = df_data.dropna(subset=[
    "vendor_name",
    "contract_value"
])

In [10]:
df_data = df_data.dropna(subset=["HPS"])

In [11]:
df_data = df_data.dropna(subset=["buyer_name"])

In [12]:
df_data = df_data[df_data["HPS"] > 0]
df_data = df_data[df_data["contract_value"] > 0]

FEATURE ENGINEERING

rasio kontrak

In [ ]:
#Rasio kontrak terhadap HPS
df_data["ratio"] = df_data["contract_value"] / df_data["HPS"]

In [14]:
#B. DISCOUNT PERCENTAGE
df_data["discount_percentage"] = (
    (df_data["HPS"] - df_data["contract_value"])
    / df_data["HPS"]
) * 100

In [16]:
# C. VENDOR WIN COUNT
# =========================================================
# Berapa kali vendor menang tender

vendor_win_count = (
    df_data.groupby("vendor_name")
    .size()
    .reset_index(name="vendor_win_count")
)
# merge ke dataframe utama
df_data = df_data.merge(
    vendor_win_count,
    on="vendor_name",
    how="left"
)


In [18]:
# =========================================================
# D. VENDOR-INSTANSI FREQUENCY
# =========================================================
# Seberapa sering vendor menang di instansi tertentu

vendor_instansi_frequency = (
    df_data.groupby(["vendor_name", "buyer_name"])
    .size()
    .reset_index(name="vendor_instansi_frequency")
)

# merge kembali
df_data = df_data.merge(
    vendor_instansi_frequency,
    on=["vendor_name", "buyer_name"],
    how="left"
)

In [19]:
# =========================================================
# E. AVERAGE VENDOR RATIO
# =========================================================
# Rata-rata ratio tiap vendor

avg_vendor_ratio = (
    df_data.groupby("vendor_name")["ratio"]
    .mean()
    .reset_index(name="avg_vendor_ratio")
)

# merge kembali
df_data = df_data.merge(
    avg_vendor_ratio,
    on="vendor_name",
    how="left"
)

In [20]:
# =========================================================
# OPTIONAL: ROUND AGAR RAPI
# =========================================================

df_data["ratio"] = df_data["ratio"].round(4)

df_data["discount_percentage"] = (
    df_data["discount_percentage"].round(2)
)

df_data["avg_vendor_ratio"] = (
    df_data["avg_vendor_ratio"].round(4)
)

In [22]:
# =========================================================
# CEK HASIL FEATURE ENGINEERING
# =========================================================

print("Feature engineering selesai.\n")

print(df_data[
    [
        "vendor_name",
        "buyer_name",
        "HPS",
        "contract_value",
        "ratio",
        "discount_percentage",
        "vendor_win_count",
        "vendor_instansi_frequency",
        "avg_vendor_ratio"
    ]
].head())

# =========================================================
# SAVE FEATURED DATASET
# =========================================================

df_data.to_csv(
    "featured_procurement_dataset.csv",
    index=False
)

print("\nDataset berhasil disimpan.")

Feature engineering selesai.

                  vendor_name                       buyer_name           HPS  \
0        PT. DUTA BHUANA JAYA  Pemerintah Daerah Kota Surabaya  7.445028e+08   
1            MARGA PERKASA,CV  Pemerintah Daerah Kota Surabaya  4.793422e+09   
2   CV. Naga Kencana Wiratama  Pemerintah Daerah Kota Surabaya  2.255991e+09   
3  CV. Tiga Points Jaya Karya  Pemerintah Daerah Kota Surabaya  3.425503e+09   
4             CV. Citra Karya  Pemerintah Daerah Kota Surabaya  1.307263e+09   

   contract_value   ratio  discount_percentage  vendor_win_count  \
0    6.666604e+08  0.8954                10.46                97   
1    4.129909e+09  0.8616                13.84                 2   
2    2.012700e+09  0.8922                10.78                 3   
3    2.566207e+09  0.7491                25.09                 1   
4    1.231890e+09  0.9423                 5.77                 1   

   vendor_instansi_frequency  avg_vendor_ratio  
0                          6   

In [23]:
print(df_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 161422 entries, 0 to 161421
Data columns (total 13 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   ocid                       161422 non-null  str    
 1   buyer_name                 161422 non-null  str    
 2   vendor_name                161422 non-null  str    
 3   HPS                        161422 non-null  float64
 4   contract_value             161422 non-null  float64
 5   category                   161422 non-null  str    
 6   item_description           161422 non-null  str    
 7   date                       161422 non-null  str    
 8   ratio                      161422 non-null  float64
 9   discount_percentage        161422 non-null  float64
 10  vendor_win_count           161422 non-null  int64  
 11  vendor_instansi_frequency  161422 non-null  int64  
 12  avg_vendor_ratio           161422 non-null  float64
dtypes: float64(5), int64(2), str(6)
memory u